In [ ]:
 from google.colab import files

uploaded = files.upload()

Saving voice 1.ogg to voice 1.ogg


In [ ]:
!pip install openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 6.0 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=43086273c9b2c3461141a0068b59f60e45800818e3e7adca3a306c63e99a0ab1
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [ ]:
!whisper "voice 1.ogg" --model base


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
Detecting language using up to the first 30 seconds. Use `--language` to specify the language
Detected language: English
[00:00.000 --> 00:03.000]  We'll show all employees name.


In [ ]:
with open("voice 1.txt", "r") as f:
    transcribed_text = f.read()

print("Voice Text:")
print(transcribed_text)

Voice Text:
We'll show all employees name.



In [ ]:
import sqlite3

conn = sqlite3.connect("company.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS employees(
id INTEGER PRIMARY KEY,
name TEXT,
department TEXT,
salary INTEGER
)
""")

In [ ]:
employees = [
    (1,"Arun","IT",70000),
    (2,"Priya","HR",50000),
    (3,"Kumar","IT",90000),
    (4,"Divya","Finance",65000),
    (5,"Ravi","IT",80000)
]

cursor.executemany(
    "INSERT OR REPLACE INTO employees VALUES (?, ?, ?, ?)",
    employees
)

conn.commit()

In [ ]:
!pip install -U google-genai

In [ ]:
from google.colab import userdata
API_KEY = userdata.get('GEMINI_API_KEY')

In [ ]:
user_question = transcribed_text

In [ ]:
prompt = f"""
You are an SQLite expert.

Database Schema:

employees(
id INTEGER,
name TEXT,
department TEXT,
salary INTEGER
)

Convert the user's request into SQLite SQL.

Return ONLY SQL.

Question:
{user_question}
"""

In [ ]:
from google import genai

client = genai.Client(api_key=API_KEY)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

sql_query = response.text.strip()

In [ ]:
import re

# Remove common markdown code block indicators and strip whitespace
sql_query = re.sub(r"```(?:sql)?", "", sql_query)
sql_query = re.sub(r"```", "", sql_query)
sql_query = sql_query.strip()

# If after stripping, it still doesn't start with SELECT, try to find SELECT
# and extract the query from there. This handles cases where the LLM might
# add other text or miss a markdown fence (like the 'ite' seen in previous runs).
if not sql_query.upper().startswith("SELECT"):
    select_keyword_index = sql_query.upper().find("SELECT")
    if select_keyword_index != -1:
        sql_query = sql_query[select_keyword_index:].strip()
    # If 'SELECT' is not found at all, the query is genuinely not a SELECT query,
    # and the subsequent validation in 80f99350 will catch it as intended.

In [ ]:
if not sql_query.upper().startswith("SELECT"):
    raise Exception("Only SELECT queries allowed")

In [ ]:
import pandas as pd

try:
    result = pd.read_sql_query(sql_query, conn)
    print("Query ran successfully!")
except Exception as e:
    print(f"SQL Error: {e}")
    result = None

if result is not None:
    display(result)
else:
    print("No results to display.")


Query ran successfully!


,name
0,Arun
1,Priya
2,Kumar
3,Divya
4,Ravi


In [ ]:
print("Generated SQL:")
print(sql_query)

Generated SQL:
SELECT name FROM employees;


In [ ]:
print("Voice Input:")
print(transcribed_text)

print("\nGenerated SQL:")
print(sql_query)

print("\nResults:")
display(result)

Voice Input:
We'll show all employees name.


Generated SQL:
SELECT name FROM employees;

Results:


,name
0,Arun
1,Priya
2,Kumar
3,Divya
4,Ravi


In [ ]:
#  Always close the connection when done
conn.close()
print("Database connection closed.")

Database connection closed.
